# A2 Q3 - NRMS baseline reproduced, then beaten (run this notebook on Kaggle)

All four parts of Q3, end to end, in one notebook:

1. **Reproduce the baseline.** `NRMSDocVec` from
   [`ebanalyse/ebnerd-benchmark`](https://github.com/ebanalyse/ebnerd-benchmark),
   the authors' own model code, cloned at a pinned commit and imported
   unmodified (the test cell asserts `git status --porcelain` is empty), run
   on **both** datasets - `ebnerd_large` and `mind_large` - through A1's
   unified schema and A1 Q3's document embeddings, which is exactly the
   input `NRMSDocVec` is designed for.
2. **Improve it with one principled change.** A recency prior on the user
   encoder's history attention: real elapsed time on EB-NeRD, an ordinal
   proxy on MIND, the same two bases A2 Q1 already uses for its behavioural
   features. Only `_build_userencoder` changes; the news encoder, the
   hyperparameters, the training data and the parameter count are identical.
3. **Ablation.** Three variants, not two. The recency layer also masks the
   zero-padded history slots the baseline attends over, so a two-way
   comparison would credit the recency signal for a padding fix. The middle
   variant `masked_uniform` is the same architecture fed all-zero
   log-weights, which isolates the two effects.
4. **Statistical significance.** Paired bootstrap 95% CIs on every claimed
   difference, over per-impression metric values on identical impressions,
   using the same estimator as `evaluation.paired_bootstrap_ci`.

**Inputs**: attach the Kaggle Dataset built by `src/nrms_inputs.ipynb`
(`data/kaggle_nrms/`). Whichever datasets are present are auto-discovered,
so a session can be scoped to one dataset by attaching only its files.

**Settings**: Accelerator -> GPU (T4 x2 or P100), Internet -> On (the clone
and a `tf-keras` install need it).

**Outputs** to `/kaggle/working/`: `nrms_metrics_{dataset}.json`,
`nrms_{variant}_{dataset}.weights.h5`, `nrms_ablation.png`,
`nrms_paired_ci.png`.

The evaluation population is the *same* 200,000 impressions per split that
A2 Q2 measured BM25, the embedding retriever and the LightGBM re-ranker on
(`nrms_inputs.ipynb` draws it with Q2's own function at Q2's seed), so the
numbers here can be read directly against `reranker_eval_metrics.json`.

In [ ]:
import os
import subprocess
import sys

# ebrec's layers.py is written against the Keras 2 API: K.dot,
# K.permute_dimensions, K.one_hot and friends were all removed in Keras 3, so
# the authors' baseline can only run as they wrote it under legacy Keras.
# This has to be set before tensorflow is imported for the first time.
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

EBNERD_REPO_URL = "https://github.com/ebanalyse/ebnerd-benchmark.git"
# Pinned so "the baseline" means one specific revision of their code.
EBNERD_COMMIT = "5164e2ce7c92b99cbcb853d5f804cc95f0232b2f"
# Cloned outside /kaggle/working on purpose: everything under working is
# packaged as the notebook's downloadable output, and the repo carries ~2GB
# of example plots and notebooks that have no business in it.
REPO_DIR = "/tmp/ebnerd-benchmark"


def run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


try:
    import tf_keras  # noqa: F401
except ImportError:
    run([sys.executable, "-m", "pip", "install", "-q", "tf-keras"])

try:
    import polars as pl
except ImportError:
    run([sys.executable, "-m", "pip", "install", "-q", "polars"])
    import polars as pl

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "clone", "--quiet", EBNERD_REPO_URL, REPO_DIR])
run(["git", "-C", REPO_DIR, "checkout", "--quiet", EBNERD_COMMIT])

# The model code is imported from the clone rather than `pip install .`:
# ebrec pins polars==0.20.8, numpy<1.26.1, torch<2.3 and transformers<4.37.3,
# none of which this notebook needs and all of which would either downgrade
# Kaggle's stack underneath a running kernel or fail to resolve at all. The
# two modules actually imported (nrms_docvec.py, layers.py) depend on nothing
# but tensorflow and numpy. Their *dataloader* is the piece that needs old
# polars - `map_list_article_id_to_value` calls `Expr.replace(default=...)`,
# removed in polars 1.0 - so the batching is reimplemented below and checked
# against their semantics instead.
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import glob
import json
import math
import time
from datetime import timezone, datetime
from pathlib import Path

import numpy as np
import tensorflow as tf

# layers.py does `import tensorflow.keras as keras`. With legacy Keras active
# tf.keras resolves to tf_keras, but the `tensorflow.keras` module *path* is a
# lazy-loader shim whose importability varies across TF versions; registering
# it in sys.modules makes that import statement resolve to the same module
# object regardless of version.
sys.modules.setdefault("tensorflow.keras", tf.keras)

from ebrec.models.newsrec.layers import AttLayer2, SelfAttention  # noqa: E402
from ebrec.models.newsrec.model_config import (  # noqa: E402
    DEFAULT_DOCUMENT_SIZE,
    hparams_nrms_docvec,
    hparams_to_dict,
)
from ebrec.models.newsrec.nrms_docvec import NRMSDocVec  # noqa: E402

K = tf.keras.backend
WORK = Path("/kaggle/working")

# --- constants -------------------------------------------------------------
SEED = 42

# Held at A1's RECENT_N_CLICKS, which is also ebrec's own
# hparams_nrms_docvec.history_size default: the baseline runs at its
# published setting and every method in this project reads the same window.
HISTORY_SIZE = 20

# Wu et al. (2019) negative sampling, ebrec's own default ratio.
NPRATIO = 4
BATCH_SIZE_TRAIN = 32  # ebrec's args_nrms_docvec default
EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2
LR_PLATEAU_PATIENCE = 1

# Recency half-lives, identical to A2 Q1 #3's feature definitions.
HALF_LIFE_HOURS = 72.0  # EB-NeRD: real elapsed time
HALF_LIFE_CLICKS = 5.0  # MIND: ordinal proxy, no timestamps exist
# exp(-inf) is a NaN generator; a large finite log-weight gives exactly zero
# attention after exponentiation without one.
PAD_LOG_WEIGHT = -1.0e9

USER_BATCH = 1024  # impressions per user-encoder forward pass
NEWS_BATCH = 4096  # articles per news-encoder forward pass

BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 0
NDCG_K_VALUES = [5, 10]
METRIC_NAMES = ["auc", "mrr"] + [f"ndcg{k}" for k in NDCG_K_VALUES]

# baseline: the authors' model, untouched.
# masked_uniform: our architecture, all-zero log-weights (padding masked only).
# recency: our architecture, real recency log-weights. The improvement.
VARIANTS = ["baseline", "masked_uniform", "recency"]
SPLITS = ["val", "test"]

# Optional cap on training samples, for a session that has to fit a tighter
# GPU budget. None means "use every sampled impression".
TRAIN_SAMPLE_CAP = None

np.random.seed(SEED)
tf.random.set_seed(SEED)


def keras_version() -> str | None:
    """Under TF_USE_LEGACY_KERAS, tf.keras resolves to the shim module
    tf_keras.api._v2.keras, which carries no __version__ of its own, so ask
    the tf_keras package (then standalone keras) and report None rather than
    raising if neither answers. This string is diagnostic only: the
    authoritative check that the Keras 2 API is present is the
    backend-function probe in the test cell below."""
    version = getattr(tf.keras, "__version__", None)
    if version is None:
        for module_name in ("tf_keras", "keras"):
            try:
                version = __import__(module_name).__version__
                break
            except Exception:  # noqa: BLE001
                continue
    return str(version) if version else None


KERAS_VERSION = keras_version()
print("tensorflow", tf.__version__, "| keras", KERAS_VERSION or "unknown", "| polars", pl.__version__)
print("tf.keras resolves to:", tf.keras.__name__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
import inspect


def test_environment() -> None:
    # 1. Legacy Keras really is active. The authoritative test is that the
    #    Keras 2 backend functions ebrec's layers actually call are present,
    #    not a version string: under TF_USE_LEGACY_KERAS tf.keras resolves to
    #    a shim module (tf_keras.api._v2.keras) that carries no __version__ at
    #    all, so the version is asserted only when one could be determined.
    #    Without legacy Keras the imports above still succeed and the failure
    #    surfaces much later, inside the first model build.
    for fn in ("dot", "permute_dimensions", "one_hot", "squeeze", "expand_dims"):
        assert callable(getattr(K, fn, None)), (
            f"tf.keras.backend.{fn} is missing, so this is Keras 3; ebrec's layers need the "
            "Keras 2 API. TF_USE_LEGACY_KERAS must be set before tensorflow is first imported."
        )
    if KERAS_VERSION is not None:
        assert KERAS_VERSION.startswith("2."), (KERAS_VERSION, tf.keras.__name__)

    # 2. The baseline is the authors' code at the pinned revision, verbatim.
    head = subprocess.run(
        ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], check=True, capture_output=True, text=True
    ).stdout.strip()
    assert head == EBNERD_COMMIT, (head, EBNERD_COMMIT)
    dirty = subprocess.run(
        ["git", "-C", REPO_DIR, "status", "--porcelain"], check=True, capture_output=True, text=True
    ).stdout.strip()
    assert dirty == "", f"the clone has local modifications, so it is not a reproduction:\n{dirty}"
    for obj in (NRMSDocVec, AttLayer2, SelfAttention):
        src = inspect.getfile(obj)
        assert src.startswith(REPO_DIR), f"{obj.__name__} was imported from {src}, not the clone"

    # 3. Their published document-vector size is what A1's embeddings produce;
    #    asserted against the data itself in the loading cell as well.
    assert DEFAULT_DOCUMENT_SIZE == 768, DEFAULT_DOCUMENT_SIZE
    assert hparams_nrms_docvec.head_num * hparams_nrms_docvec.head_dim == 256


test_environment()
print("environment OK: legacy Keras active, clone pinned at", EBNERD_COMMIT[:8], "and unmodified")

## Load the uploaded inputs

Everything is converted to numpy/Arrow-backed arrays here and never to
Python lists: `ebnerd_large`'s history alone is ~16M article ids across
~800k users, and materialising Arrow list columns as Python objects costs
~182 bytes per element (A2 Q1 #8 measured it) - several GB for data that
occupies a few hundred MB in Arrow. The id-to-row mapping is therefore an
explode-and-left-join in polars, and the matrices come out via
`list.to_array(...).to_numpy()`, the same conversion A2 Q2 #5 measured at
39x faster than a list comprehension.

Candidate sets are ragged (2 to 299 in-view articles), so they are stored
CSR-style: one flat `cand` array plus an `offsets` array, rather than a
padded rectangle that would be mostly padding at MIND's inview sizes.

History rows are **right**-aligned with zero padding on the left, matching
`ebrec.utils._behaviors.truncate_history(padding_value=0)`: index 0 is the
unknown/padding row of the article matrix, exactly as
`create_lookup_objects` defines it.

In [ ]:
def discover_datasets() -> dict[str, dict[str, str]]:
    found: dict[str, dict[str, str]] = {}
    for path in sorted(glob.glob("/kaggle/input/**/nrms_*_train.parquet", recursive=True)):
        name = os.path.basename(path)[len("nrms_") : -len("_train.parquet")]
        directory = os.path.dirname(path)
        entry = {"train": path}
        for split in SPLITS:
            entry[split] = os.path.join(directory, f"nrms_{name}_{split}.parquet")
        entry["history"] = os.path.join(directory, f"nrms_{name}_history.parquet")
        matches = sorted(
            glob.glob(f"/kaggle/input/**/{name}_article_embeddings.parquet", recursive=True)
        )
        assert matches, f"{name}: no {name}_article_embeddings.parquet under /kaggle/input"
        entry["embeddings"] = matches[0]
        for key, value in entry.items():
            assert os.path.exists(value), f"{name}: missing {key} at {value}"
        found[name] = entry
    assert found, f"no nrms_*_train.parquet under /kaggle/input; found {glob.glob('/kaggle/input/*')}"
    return found


DATASET_PATHS = discover_datasets()
DATASETS = sorted(DATASET_PATHS)

MANIFEST = {}
for path in glob.glob("/kaggle/input/**/nrms_inputs_manifest.json", recursive=True):
    with open(path) as f:
        MANIFEST = json.load(f)
    break


def build_article_lookup(path: str) -> tuple[dict[str, int], np.ndarray, int]:
    """1-based article index plus an (n+1, dim) matrix whose row 0 is the
    zero/unknown vector - the same layout as ebrec's create_lookup_objects,
    which the test cell checks against directly."""
    emb = pl.read_parquet(path, columns=["article_id", "embedding"])
    dim = int(emb["embedding"].list.len().max())
    vectors = emb["embedding"].list.to_array(dim).to_numpy().astype(np.float32, copy=False)
    matrix = np.zeros((vectors.shape[0] + 1, dim), dtype=np.float32)
    matrix[1:] = vectors
    index = {aid: i for i, aid in enumerate(emb["article_id"].to_list(), start=1)}
    return index, matrix, dim


def index_frame(index: dict[str, int]) -> pl.DataFrame:
    return pl.DataFrame(
        {"article_id": list(index.keys()), "art_row": np.fromiter(index.values(), dtype=np.int32)}
    )


def to_padded_index_matrix(
    df: pl.DataFrame, list_col: str, idx_df: pl.DataFrame, width: int
) -> np.ndarray:
    """Map a list-of-article-id column to a right-aligned (n, width) int32
    matrix of article-matrix rows, 0 for padding or an unmapped id. Done as
    an explode + left join in Arrow rather than a Python dict lookup per
    element.

    `_flat` exists because a polars left join is not documented to preserve
    row order (SPEC.md Q1 #5 records the same trap), and here the order *is*
    the data: a scrambled history is no longer chronological, so the recency
    weights would be attached to the wrong clicks. Sorting on it before the
    group-by restores both the row order and the within-row order, since agg
    keeps the order rows appear in."""
    mapped = (
        df.lazy()
        .with_row_index("_row")
        .select("_row", list_col)
        .explode(list_col)
        .with_row_index("_flat")
        .join(idx_df.lazy(), left_on=list_col, right_on="article_id", how="left")
        .sort("_flat")
        .select("_row", pl.col("art_row").fill_null(0).cast(pl.Int32))
        .group_by("_row")
        .agg(pl.col("art_row"))
        .sort("_row")
        .select(
            pl.col("art_row")
            .list.reverse()
            .list.eval(pl.element().extend_constant(0, width))
            .list.reverse()
            .list.tail(width)
            .list.to_array(width)
        )
        .collect()
    )
    return mapped["art_row"].to_numpy().astype(np.int32, copy=False)


def load_split(path: str, idx_df: pl.DataFrame, user_row: pl.DataFrame) -> dict:
    """CSR-style candidate arrays plus per-impression user row and timestamp."""
    df = pl.read_parquet(path).join(user_row, on="user_id", how="left")
    assert df["hist_row"].null_count() == 0, f"{path}: an impression's user has no history row"
    counts = df["article_ids_inview"].list.len().to_numpy().astype(np.int64)
    offsets = np.zeros(df.height + 1, dtype=np.int64)
    np.cumsum(counts, out=offsets[1:])
    # `_flat` then `.sort("_flat")` for the same reason as
    # to_padded_index_matrix: the join may reorder rows, and here the flat
    # candidate array is addressed by `offsets`, so a reordering would hand
    # every impression somebody else's candidates.
    flat = (
        df.lazy()
        .with_row_index("_row")
        .select("_row", "article_ids_inview", "article_ids_clicked")
        .explode("article_ids_inview")
        .with_columns(
            pl.col("article_ids_inview").is_in(pl.col("article_ids_clicked")).alias("clicked")
        )
        .with_row_index("_flat")
        .join(idx_df.lazy(), left_on="article_ids_inview", right_on="article_id", how="left")
        .sort("_flat")
        .select(
            "_row",
            pl.col("art_row").fill_null(0).cast(pl.Int32),
            pl.col("clicked").cast(pl.Int8),
        )
        .collect()
    )
    return {
        "n_impressions": df.height,
        "impression_id": df["impression_id"],
        "hist_row": df["hist_row"].to_numpy().astype(np.int32),
        "impression_us": df["impression_time"].dt.epoch("us").to_numpy().astype(np.int64),
        "offsets": offsets,
        "cand": flat["art_row"].to_numpy().astype(np.int32),
        "label": flat["clicked"].to_numpy().astype(np.int8),
        "n_clicked_distinct": int(
            df.select(pl.col("article_ids_clicked").list.unique().list.len().sum()).item()
        ),
    }


def build_dataset(name: str) -> dict:
    paths = DATASET_PATHS[name]
    t0 = time.perf_counter()
    index, art_matrix, dim = build_article_lookup(paths["embeddings"])
    idx_df = index_frame(index)

    hist = pl.read_parquet(paths["history"])
    basis = "elapsed_time" if "timestamp_sequence" in hist.columns else "ordinal_proxy"
    hist_idx = to_padded_index_matrix(hist, "article_id_sequence", idx_df, HISTORY_SIZE)
    hist_len = hist["article_id_sequence"].list.len().to_numpy().astype(np.int32)
    hist_us = None
    if basis == "elapsed_time":
        hist_us = (
            hist.lazy()
            .select(
                pl.col("timestamp_sequence")
                # cast_time_unit first: casting a Datetime straight to Int64
                # yields whatever unit the column happens to carry, and a
                # millisecond column would silently divide every elapsed age
                # by 1000.
                .list.eval(pl.element().dt.cast_time_unit("us").cast(pl.Int64))
                .list.reverse()
                .list.eval(pl.element().extend_constant(0, HISTORY_SIZE))
                .list.reverse()
                .list.tail(HISTORY_SIZE)
                .list.to_array(HISTORY_SIZE)
            )
            .collect()["timestamp_sequence"]
            .to_numpy()
            .astype(np.int64, copy=False)
        )
    user_row = hist.select("user_id").with_row_index("hist_row").select(
        "user_id", pl.col("hist_row").cast(pl.Int32)
    )

    splits = {split: load_split(paths[split], idx_df, user_row) for split in ["train"] + SPLITS}
    ds = {
        "name": name,
        "dim": dim,
        "n_articles": art_matrix.shape[0] - 1,
        "art_matrix": art_matrix,
        "art_index": index,
        "hist_idx": hist_idx,
        "hist_len": hist_len,
        "hist_us": hist_us,
        "basis": basis,
        "half_life": HALF_LIFE_HOURS if basis == "elapsed_time" else HALF_LIFE_CLICKS,
        "n_users": hist.height,
        "splits": splits,
    }
    print(
        f"{name}: {ds['n_articles']:,} articles (dim {dim}), {ds['n_users']:,} users, basis {basis}, "
        f"loaded in {time.perf_counter() - t0:.1f}s"
    )
    for split, s in splits.items():
        print(
            f"   {split}: {s['n_impressions']:,} impressions, {len(s['cand']):,} candidates, "
            f"{s['label'].mean():.4f} positive rate"
        )
    return ds


datasets = {name: build_dataset(name) for name in DATASETS}
DATASETS

In [ ]:
from ebrec.utils._python import create_lookup_objects  # noqa: E402


def test_inputs() -> None:
    for name, ds in datasets.items():
        # The document vector size must match what NRMSDocVec is configured
        # for; a silent mismatch would train on a truncated news input.
        assert ds["dim"] == DEFAULT_DOCUMENT_SIZE, (name, ds["dim"])
        assert ds["art_matrix"].dtype == np.float32
        assert not np.isnan(ds["art_matrix"]).any(), f"{name}: NaN in the article matrix"
        assert np.all(ds["art_matrix"][0] == 0.0), f"{name}: row 0 is not the zero/unknown row"

        # Our columnar lookup must agree with the authors' own
        # create_lookup_objects, which is the definition the baseline's own
        # dataloader would have used - same 1-based indexing in catalogue
        # order, same zero row at index 0. Checked on the first 500 articles
        # because their implementation builds a Python dict of one-element
        # polars Series and is not affordable over a whole catalogue.
        emb = pl.read_parquet(DATASET_PATHS[name]["embeddings"], columns=["article_id", "embedding"]).head(500)
        sample = {
            aid: np.asarray(vec, dtype=np.float32)
            for aid, vec in zip(emb["article_id"].to_list(), emb["embedding"].to_list())
        }
        their_index, their_matrix = create_lookup_objects(sample, unknown_representation="zeros")
        assert their_matrix.shape == (501, ds["dim"]), their_matrix.shape
        assert np.all(their_matrix[0] == 0.0)
        for aid, series in their_index.items():
            i = int(series[0])
            assert ds["art_index"][aid] == i, (name, aid, ds["art_index"][aid], i)
            assert np.array_equal(their_matrix[i], ds["art_matrix"][i]), f"{name}: {aid} vector mismatch"

        # Padding: a zero row can only come from a padded slot, never from a
        # real article, because nrms_inputs.ipynb asserted 100% embedding
        # coverage of every referenced id.
        expected_pads = int((HISTORY_SIZE - np.minimum(ds["hist_len"], HISTORY_SIZE)).sum())
        assert int((ds["hist_idx"] == 0).sum()) == expected_pads, name
        # Right alignment: real ids occupy the tail of each row.
        rows = np.flatnonzero((ds["hist_len"] > 0) & (ds["hist_len"] < HISTORY_SIZE))[:1000]
        for r in rows:
            n = int(ds["hist_len"][r])
            assert np.all(ds["hist_idx"][r, :-n] == 0) and np.all(ds["hist_idx"][r, -n:] != 0), r

        # The columnar arrays must reproduce the source lists *in order*, not
        # merely contain the right ids. This is the assertion that catches a
        # reordering join: without the _flat sort, an impression would be
        # scored against another impression's candidates, or a history would
        # stop being chronological, and nothing else here would notice.
        raw_hist = pl.read_parquet(DATASET_PATHS[name]["history"]).head(200)
        for r in range(raw_hist.height):
            ids = raw_hist["article_id_sequence"][r].to_list()[-HISTORY_SIZE:]
            expected = [ds["art_index"][a] for a in ids]
            row = ds["hist_idx"][r]
            pad_width = HISTORY_SIZE - len(expected)
            assert row[pad_width:].tolist() == expected, (name, "history", r)
            assert np.all(row[:pad_width] == 0), (name, "history pad", r)
            if ds["hist_us"] is not None:
                assert np.all(ds["hist_us"][r, :pad_width] == 0), (name, "timestamp pad", r)
        for split in SPLITS:
            s = ds["splits"][split]
            raw = pl.read_parquet(
                DATASET_PATHS[name][split], columns=["article_ids_inview", "article_ids_clicked"]
            ).head(200)
            for i in range(raw.height):
                inview = raw["article_ids_inview"][i].to_list()
                clicked = set(raw["article_ids_clicked"][i].to_list())
                lo, hi = s["offsets"][i], s["offsets"][i + 1]
                assert s["cand"][lo:hi].tolist() == [ds["art_index"][a] for a in inview], (
                    name,
                    split,
                    i,
                )
                assert s["label"][lo:hi].tolist() == [int(a in clicked) for a in inview], (
                    name,
                    split,
                    i,
                )

        # Epoch microseconds, not milliseconds or seconds: a unit slip would
        # scale every elapsed age by 1000 and quietly flatten or sharpen the
        # whole recency prior. Year 2000 in microseconds is the floor.
        YEAR_2000_US = 946_684_800_000_000
        if ds["hist_us"] is not None:
            real_us = ds["hist_us"][ds["hist_idx"] != 0]
            assert real_us.min() > YEAR_2000_US, (name, "history timestamps", real_us.min())
        for split in SPLITS:
            assert ds["splits"][split]["impression_us"].min() > YEAR_2000_US, (name, split)

        for split in SPLITS:
            s = ds["splits"][split]
            assert s["offsets"][-1] == len(s["cand"]) == len(s["label"])
            sizes = np.diff(s["offsets"])
            assert sizes.min() > 0, f"{name}/{split}: an impression with no candidate"
            # AUC/MRR/nDCG are undefined without both a click and a non-click.
            pos = np.add.reduceat(s["label"].astype(np.int64), s["offsets"][:-1])
            assert pos.min() >= 1, f"{name}/{split}: an impression with no positive"
            assert (sizes - pos).min() >= 1, f"{name}/{split}: an impression with no negative"
            # EB-NeRD lists the same article twice in one impression's clicked
            # list when it was clicked twice (A2 Q2 #11), so the per-candidate
            # label count matches the *distinct* clicked count, not the raw one.
            assert int(pos.sum()) == s["n_clicked_distinct"], (
                name,
                split,
                int(pos.sum()),
                s["n_clicked_distinct"],
            )

        if MANIFEST:
            entry = MANIFEST["datasets"][name]
            assert entry["recency_weight_basis"] == ds["basis"], (name, entry, ds["basis"])
            assert MANIFEST["hyperparameters"]["history_size"] == HISTORY_SIZE
            for split in SPLITS:
                assert entry["impressions"][split] == ds["splits"][split]["n_impressions"]
        else:
            print(f"   {name}: no manifest attached, basis inferred from the history schema")


test_inputs()
print("inputs OK: lookup matches ebrec's own, padding right-aligned, every impression gradeable")

## Recency log-weights

The improvement's whole input. For history click *i* scored at impression
time *t*:

```
log w_i = -ln2 * age_i / half_life      (then shifted so max log w = 0)
```

- **EB-NeRD** (`elapsed_time`): `age_i` is real elapsed hours,
  `t - timestamp_i`, with `HALF_LIFE_HOURS = 72`.
- **MIND** (`ordinal_proxy`): MIND's raw data has no per-click timestamps at
  all, ever (A1 Q1 #2) - only click order - so `age_i` is the click's
  position from the most recent one and `HALF_LIFE_CLICKS = 5`.

Both match A2 Q1 #3's feature definitions exactly, rather than introducing a
third notion of recency into the project.

Two details that matter:

- Weights are per **(impression, click)**, not per user. A user's history is
  a fixed pre-window snapshot, but a click three hours before the first
  evaluated impression has decayed much further by that user's last test
  impression days later; a per-user weight vector would misprice that.
- **Log**-weights, never weights. The layer needs `log w` anyway (it adds
  them to the attention logits), and computing them directly avoids
  `exp` underflow on EB-NeRD's older clicks entirely. Subtracting each
  row's maximum is exactly invariant, because the attention weights are
  renormalised immediately afterwards.

In [ ]:
LN2 = float(np.log(2.0))
US_PER_HOUR = 3.6e9


def history_log_weights(ds: dict, hist_rows: np.ndarray, impression_us: np.ndarray) -> np.ndarray:
    """(len(hist_rows), HISTORY_SIZE) float32 log-weights for the `recency`
    variant. Padded slots get PAD_LOG_WEIGHT, so they receive exactly zero
    attention."""
    idx = ds["hist_idx"][hist_rows]
    pad = idx == 0
    if ds["basis"] == "elapsed_time":
        ages = (impression_us[:, None] - ds["hist_us"][hist_rows]).astype(np.float64) / US_PER_HOUR
        # A history click at or after the impression would be leakage; A1
        # verified 0 violations on real data, and clipping keeps a
        # hypothetical one from becoming a positive (age-boosting) weight.
        np.maximum(ages, 0.0, out=ages)
    else:
        # Position from the most recent real click. History is right-aligned,
        # so slot j is (HISTORY_SIZE - 1 - j) clicks back.
        ranks = (HISTORY_SIZE - 1 - np.arange(HISTORY_SIZE)).astype(np.float64)
        ages = np.broadcast_to(ranks, idx.shape)
    log_w = -LN2 * ages / ds["half_life"]
    log_w[pad] = -np.inf
    # Shift so the most recent real click sits at 0: invariant after
    # renormalisation, and it keeps exp() away from underflow.
    row_max = np.max(log_w, axis=1, keepdims=True)
    row_max[~np.isfinite(row_max)] = 0.0  # cold-start user: no real click at all
    log_w = log_w - row_max
    log_w[pad] = PAD_LOG_WEIGHT
    return log_w.astype(np.float32)


def uniform_log_weights(ds: dict, hist_rows: np.ndarray, impression_us: np.ndarray) -> np.ndarray:
    """The ablation control: same architecture, no recency signal. Real slots
    get 0 (uniform attention prior), padded slots PAD_LOG_WEIGHT."""
    pad = ds["hist_idx"][hist_rows] == 0
    log_w = np.zeros(pad.shape, dtype=np.float32)
    log_w[pad] = PAD_LOG_WEIGHT
    return log_w


LOG_WEIGHT_FN = {"masked_uniform": uniform_log_weights, "recency": history_log_weights}

In [ ]:
def test_log_weights() -> None:
    for name, ds in datasets.items():
        s = ds["splits"]["val"]
        rows = np.arange(min(4096, s["n_impressions"]))
        hist_rows = s["hist_row"][rows]
        log_w = history_log_weights(ds, hist_rows, s["impression_us"][rows])
        uni = uniform_log_weights(ds, hist_rows, s["impression_us"][rows])
        pad = ds["hist_idx"][hist_rows] == 0

        assert log_w.dtype == np.float32 and log_w.shape == (len(rows), HISTORY_SIZE)
        assert np.all(log_w[pad] == PAD_LOG_WEIGHT) and np.all(uni[pad] == PAD_LOG_WEIGHT)
        assert np.all(uni[~pad] == 0.0), "the control variant must carry no recency signal"
        assert np.isfinite(log_w).all(), "no infinities may reach the graph"
        # Newest real click is the reference point; nothing may exceed it.
        if (~pad).any():
            assert log_w[~pad].max() <= 0.0 + 1e-6
        rows_with_history = np.flatnonzero(ds["hist_len"][hist_rows] > 0)
        assert np.allclose(log_w[rows_with_history, -1], 0.0, atol=1e-6), (
            "the most recent click must be the row maximum"
        )
        # Older clicks decay: within the real slots, log-weights are
        # non-decreasing left to right (oldest to newest).
        real = np.where(pad, np.nan, log_w)
        diffs = np.diff(real, axis=1)
        assert np.all(np.nan_to_num(diffs, nan=0.0) >= -1e-5), "log-weights are not monotone in recency"

        if ds["basis"] == "ordinal_proxy":
            # One half-life back weighs exactly half the newest click.
            full = np.flatnonzero(ds["hist_len"][hist_rows] >= HISTORY_SIZE)[:1]
            if len(full):
                r = full[0]
                back = int(HALF_LIFE_CLICKS)
                assert abs(float(np.exp(log_w[r, -1 - back])) - 0.5) < 1e-6, log_w[r, -1 - back]
        else:
            # Hand-computed against the raw timestamps for one impression.
            multi = np.flatnonzero(ds["hist_len"][hist_rows] > 1)
            assert len(multi), f"{name}: no sampled impression has more than one history click"
            r = int(multi[0])
            hr = hist_rows[r]
            imp_us = float(s["impression_us"][rows][r])
            age = (imp_us - float(ds["hist_us"][hr, -2])) / US_PER_HOUR
            newest_age = (imp_us - float(ds["hist_us"][hr, -1])) / US_PER_HOUR
            expected = -LN2 * (max(age, 0.0) - max(newest_age, 0.0)) / HALF_LIFE_HOURS
            assert abs(float(log_w[r, -2]) - expected) < 1e-4, (log_w[r, -2], expected)

        n_cold = int((ds["hist_len"][hist_rows] == 0).sum())
        print(f"   {name}: {n_cold} cold-start impressions in the sample, all slots padded for them")


test_log_weights()
print("recency log-weights OK (per impression, monotone, padding excluded, half-life exact)")

## Training pairs: Wu et al. (2019) negative sampling

One training sample per **distinct** clicked article: that positive plus
`NPRATIO` negatives drawn with replacement from the same impression's
non-clicked candidates, then the group is shuffled and the label is a
one-hot over `NPRATIO + 1` positions - which is what `NRMSDocVec`'s softmax
output and `categorical_crossentropy` loss expect.

This reimplements `ebrec.utils._behaviors.sampling_strategy_wu2019` in numpy
(their version needs polars 0.20, see the setup cell) with the same
semantics: positives preserved, negatives drawn from in-view minus clicked,
with replacement, at a fixed seed. The test cell runs their function on a
small fixture and checks ours produces the same row count and the same
positive/negative membership.

Early stopping runs on the **last calendar day** of this training sample,
mirroring `ebnerd_nrms_docvec.py`'s own `last_dt` split. Our `val` split is
therefore never used for model selection: it is reported, not tuned against.

In [ ]:
def build_training_samples(ds: dict, seed: int = SEED) -> dict:
    s = ds["splits"]["train"]
    rng = np.random.default_rng(seed)
    offsets, cand, label = s["offsets"], s["cand"], s["label"]

    sample_hist_row: list[int] = []
    sample_us: list[int] = []
    sample_cand: list[np.ndarray] = []
    sample_label: list[np.ndarray] = []
    n_skipped = 0

    for i in range(s["n_impressions"]):
        lo, hi = offsets[i], offsets[i + 1]
        labels_i = label[lo:hi]
        cand_i = cand[lo:hi]
        positives = np.unique(cand_i[labels_i == 1])
        negatives = cand_i[labels_i == 0]
        if len(positives) == 0 or len(negatives) == 0:
            n_skipped += 1
            continue
        for pos in positives:
            drawn = negatives[rng.integers(0, len(negatives), size=NPRATIO)]
            group = np.concatenate([[pos], drawn])
            onehot = np.zeros(NPRATIO + 1, dtype=np.float32)
            onehot[0] = 1.0
            order = rng.permutation(NPRATIO + 1)
            sample_cand.append(group[order])
            sample_label.append(onehot[order])
            sample_hist_row.append(s["hist_row"][i])
            sample_us.append(s["impression_us"][i])

    out = {
        "hist_row": np.asarray(sample_hist_row, dtype=np.int32),
        "impression_us": np.asarray(sample_us, dtype=np.int64),
        "cand": np.asarray(sample_cand, dtype=np.int32),
        "label": np.asarray(sample_label, dtype=np.float32),
        "n_skipped_impressions": n_skipped,
    }
    if TRAIN_SAMPLE_CAP is not None and len(out["cand"]) > TRAIN_SAMPLE_CAP:
        keep = np.sort(rng.permutation(len(out["cand"]))[:TRAIN_SAMPLE_CAP])
        for key in ("hist_row", "impression_us", "cand", "label"):
            out[key] = out[key][keep]
    return out


def split_by_last_day(ds: dict, samples: dict) -> tuple[np.ndarray, np.ndarray]:
    """Fit / early-stopping split by impression time, the same rule as
    ebnerd_nrms_docvec.py: everything before the sample's last calendar day
    fits, the last day monitors. A dataset whose train split spans a single
    day (the demo tracks) would make one side empty, so that degenerate case
    falls back to a time-ordered 90/10 cut - still chronological, never
    random, so no future impression trains the model."""
    us = samples["impression_us"]
    day = us // (24 * 3_600_000_000)
    cutoff = day.max()
    fit_idx = np.flatnonzero(day < cutoff)
    stop_idx = np.flatnonzero(day >= cutoff)
    if len(fit_idx) == 0 or len(stop_idx) == 0:
        threshold = np.quantile(us, 0.9)
        fit_idx = np.flatnonzero(us < threshold)
        stop_idx = np.flatnonzero(us >= threshold)
        print("   train split spans one day: fell back to a chronological 90/10 cut")
    return fit_idx, stop_idx


training = {}
for name, ds in datasets.items():
    t0 = time.perf_counter()
    samples = build_training_samples(ds)
    fit_idx, stop_idx = split_by_last_day(ds, samples)
    training[name] = {"samples": samples, "fit_idx": fit_idx, "stop_idx": stop_idx}
    print(
        f"{name}: {len(samples['cand']):,} training samples "
        f"({len(fit_idx):,} fit / {len(stop_idx):,} early-stopping), "
        f"{samples['n_skipped_impressions']:,} impressions skipped for having no negative, "
        f"built in {time.perf_counter() - t0:.1f}s"
    )
{name: len(training[name]["samples"]["cand"]) for name in DATASETS}

In [ ]:
def test_training_samples() -> None:
    for name, ds in datasets.items():
        samples = training[name]["samples"]
        n = len(samples["cand"])
        assert n > 0
        assert samples["cand"].shape == (n, NPRATIO + 1)
        assert samples["label"].shape == (n, NPRATIO + 1)
        # Exactly one positive per group: this is what the softmax head and
        # categorical_crossentropy are defined against.
        assert np.all(samples["label"].sum(axis=1) == 1.0)
        assert np.all(samples["cand"] > 0), "a padded/unknown article ended up as a candidate"

        # The positive really is a click of that impression, and the negatives
        # really are not, checked against the source split.
        s = ds["splits"]["train"]
        by_hist: dict[int, int] = {}
        for i in range(min(s["n_impressions"], 20000)):
            by_hist.setdefault(int(s["hist_row"][i]), i)
        checked = 0
        for r in range(min(n, 2000)):
            i = by_hist.get(int(samples["hist_row"][r]))
            if i is None or s["impression_us"][i] != samples["impression_us"][r]:
                continue
            lo, hi = s["offsets"][i], s["offsets"][i + 1]
            clicked = set(s["cand"][lo:hi][s["label"][lo:hi] == 1].tolist())
            pos_slot = int(np.argmax(samples["label"][r]))
            assert int(samples["cand"][r, pos_slot]) in clicked, (name, r)
            for slot in range(NPRATIO + 1):
                if slot == pos_slot:
                    continue
                assert int(samples["cand"][r, slot]) not in clicked, (name, r, slot)
            checked += 1
        assert checked > 0, f"{name}: could not verify any sampled group against its impression"

        # Reproducible at a fixed seed, or the ablation compares models fitted
        # on different data.
        again = build_training_samples(ds)
        assert np.array_equal(again["cand"], samples["cand"])
        assert np.array_equal(again["label"], samples["label"])

        # The two training subsets are time-disjoint.
        fit_idx, stop_idx = training[name]["fit_idx"], training[name]["stop_idx"]
        assert len(fit_idx) > 0 and len(stop_idx) > 0, f"{name}: degenerate fit/early-stopping split"
        assert samples["impression_us"][fit_idx].max() < samples["impression_us"][stop_idx].min()

    # Cross-check our reimplementation against the authors' own sampler on a
    # fixture: 2 + 1 + 1 distinct clicks must give 4 groups of NPRATIO + 1
    # with exactly one positive each, which is the contract ours implements.
    # Skipped rather than failed if their code cannot run on the installed
    # polars, since sampling_strategy_wu2019 reaches Expr.replace-era APIs.
    fixture = pl.DataFrame(
        {
            "impression_id": ["i1", "i2", "i3"],
            "user_id": ["u1", "u1", "u2"],
            "article_ids_inview": [["a", "b", "c", "d"], ["a", "b", "c", "d", "e"], ["a", "b", "c"]],
            "article_ids_clicked": [["a", "b"], ["c"], ["a"]],
        }
    )
    theirs = None
    try:
        from ebrec.utils._behaviors import create_binary_labels_column, sampling_strategy_wu2019

        theirs = fixture.pipe(
            sampling_strategy_wu2019, npratio=NPRATIO, shuffle=True, with_replacement=True, seed=SEED
        ).pipe(create_binary_labels_column)
    except Exception as exc:  # noqa: BLE001
        print(f"   skipped the ebrec sampler cross-check ({type(exc).__name__}: {exc})")
    # Asserted outside the try, so a real disagreement fails the cell instead
    # of being reported as a skipped check.
    if theirs is not None:
        assert theirs.height == 4, theirs.height
        assert theirs["labels"].list.sum().to_list() == [1, 1, 1, 1]
        assert theirs["article_ids_inview"].list.len().to_list() == [NPRATIO + 1] * 4
        print(f"   ebrec's own sampler agrees on the fixture: {theirs.height} groups of {NPRATIO + 1}")


test_training_samples()
print("training samples OK (one-hot groups, verified positives/negatives, reproducible, time-disjoint)")

## The baseline: `NRMSDocVec`, the authors' code

`hparams_nrms_docvec` is left at its published values (16 heads x 16 dims,
200-unit attention, `[512, 512, 512]` news encoder, dropout 0.2, Adam at
1e-4, categorical cross-entropy) with only `title_size` and `history_size`
bound to our data. Every variant below is fitted with the identical
hyperparameters, so the ablation varies one thing.

In [ ]:
def make_hparams(dim: int):
    hparams_nrms_docvec.title_size = dim
    hparams_nrms_docvec.history_size = HISTORY_SIZE
    return hparams_nrms_docvec


def build_baseline(ds: dict) -> NRMSDocVec:
    return NRMSDocVec(hparams=make_hparams(ds["dim"]), seed=SEED)


_probe = build_baseline(datasets[DATASETS[0]])
print(json.dumps(hparams_to_dict(make_hparams(datasets[DATASETS[0]]["dim"])), indent=2, default=str))
print("baseline trainable parameters:", f"{_probe.model.count_params():,}")

In [ ]:
def test_baseline_graph() -> None:
    dim = datasets[DATASETS[0]]["dim"]
    # Two inputs for training (history, candidates), two for scoring
    # (history, one candidate) - the shapes NRMSDocVec documents.
    assert len(_probe.model.inputs) == 2, [t.shape for t in _probe.model.inputs]
    assert tuple(_probe.model.inputs[0].shape) == (None, HISTORY_SIZE, dim)
    assert tuple(_probe.model.inputs[1].shape) == (None, None, dim)
    assert len(_probe.scorer.inputs) == 2
    assert tuple(_probe.scorer.inputs[1].shape) == (None, 1, dim)
    assert tuple(_probe.scorer.outputs[0].shape) == (None, 1)
    # A forward pass must produce a distribution over the candidate set.
    his = np.zeros((2, HISTORY_SIZE, dim), dtype=np.float32)
    cands = np.random.default_rng(0).normal(size=(2, NPRATIO + 1, dim)).astype(np.float32)
    preds = _probe.model.predict([his, cands], verbose=0)
    assert preds.shape == (2, NPRATIO + 1)
    assert np.allclose(preds.sum(axis=1), 1.0, atol=1e-5), preds.sum(axis=1)


test_baseline_graph()
print("baseline graph OK (softmax over the candidate group, scorer emits one score per candidate)")

## The improvement: a recency prior on the history attention

`NRMSDocVec` pools a user's history with multi-head self-attention followed
by additive attention (`AttLayer2`). That pooling is **order-blind and
time-blind**: permuting a user's history leaves the user vector unchanged,
so a click from twenty minutes ago and one from three weeks ago compete
purely on content. News consumption is not like that, and the rest of this
project already exploits it - `freshness_hours` was the single largest
feature by split gain in A2 Q2's re-ranker on EB-NeRD (48.1%).

The change is one term. `AttLayer2` computes
`a_i = exp(q . tanh(W h_i + b))`, normalised over the history. The variant
adds the recency log-weight to that exponent:

```
a_i = exp(q . tanh(W h_i + b) + log w_i)      i.e.   a_i = w_i * exp(logit_i)
```

so recency multiplies the learned attention instead of replacing it, and
`log w_i = 0` for every slot recovers the authors' layer **bit for bit**
(asserted below). Nothing else moves: the news encoder, the self-attention
block, the scorer, the loss and the parameter count are all unchanged -
`RecencyAttLayer2` adds no weights, so all three variants have identical
capacity and any measured difference is the signal, not model size.

It also fixes something incidental. A padded history slot is a zero
document vector, which `AttLayer2` still assigns positive attention to,
because `exp(logit)` of a zero vector is not zero. With
`log w = -1e9` those slots drop out exactly. That is a second effect, and
crediting the recency prior for it would be wrong, which is why
`masked_uniform` exists: it is this same architecture with all-zero
log-weights on the real slots, so

- `masked_uniform - baseline` isolates the padding mask, and
- `recency - masked_uniform` isolates the recency signal itself.

In [ ]:
class RecencyAttLayer2(AttLayer2):
    """AttLayer2 with an additive log-weight prior on the attention logits.

    Inputs: `[vectors, log_weights]`, shapes (batch, seq, dim) and
    (batch, seq). Identical to the parent when `log_weights` is all zeros -
    same weights, same initialisers, same epsilon-guarded normalisation, and
    the same `+ K.epsilon()` in the denominator rather than a numerically
    "improved" softmax, so the reduction to the baseline is exact rather than
    approximate.
    """

    def build(self, input_shape):
        # The parent asserts a single rank-3 shape and sizes W/b/q from it;
        # hand it the vector branch so the weight shapes stay identical.
        super().build(input_shape[0])

    def call(self, inputs, mask=None, **kwargs):
        vectors, log_weights = inputs
        attention = K.tanh(K.dot(vectors, self.W) + self.b)
        attention = K.dot(attention, self.q)
        attention = K.squeeze(attention, axis=2)
        attention = K.exp(attention + log_weights)
        attention_weight = attention / (K.sum(attention, axis=-1, keepdims=True) + K.epsilon())
        attention_weight = K.expand_dims(attention_weight)
        return K.sum(vectors * attention_weight, axis=1)

    def compute_output_shape(self, input_shape):
        return super().compute_output_shape(input_shape[0])


class NRMSDocVecRecency(NRMSDocVec):
    """NRMSDocVec with a recency-weighted user encoder. Subclassed so the news
    encoder, loss, optimiser and hyperparameter handling are inherited from
    the authors' class rather than restated."""

    def _build_userencoder(self, titleencoder):
        his_input_title = tf.keras.Input(
            shape=(self.hparams.history_size, self.hparams.title_size), dtype="float32"
        )
        his_input_logw = tf.keras.Input(shape=(self.hparams.history_size,), dtype="float32")
        click_title_presents = tf.keras.layers.TimeDistributed(titleencoder)(his_input_title)
        y = SelfAttention(self.hparams.head_num, self.hparams.head_dim, seed=self.seed)(
            [click_title_presents] * 3
        )
        user_present = RecencyAttLayer2(self.hparams.attention_hidden_dim, seed=self.seed)(
            [y, his_input_logw]
        )
        return tf.keras.Model([his_input_title, his_input_logw], user_present, name="user_encoder")

    def _build_nrms(self):
        his_input_title = tf.keras.Input(
            shape=(self.hparams.history_size, self.hparams.title_size), dtype="float32"
        )
        his_input_logw = tf.keras.Input(shape=(self.hparams.history_size,), dtype="float32")
        pred_input_title = tf.keras.Input(shape=(None, self.hparams.title_size), dtype="float32")
        pred_input_title_one = tf.keras.Input(shape=(1, self.hparams.title_size), dtype="float32")
        pred_title_one_reshape = tf.keras.layers.Reshape((self.hparams.title_size,))(
            pred_input_title_one
        )

        titleencoder = self._build_newsencoder(
            units_per_layer=self.hparams.newsencoder_units_per_layer
        )
        self.userencoder = self._build_userencoder(titleencoder)
        self.newsencoder = titleencoder

        user_present = self.userencoder([his_input_title, his_input_logw])
        news_present = tf.keras.layers.TimeDistributed(self.newsencoder)(pred_input_title)
        news_present_one = self.newsencoder(pred_title_one_reshape)

        preds = tf.keras.layers.Activation(activation="softmax")(
            tf.keras.layers.Dot(axes=-1)([news_present, user_present])
        )
        pred_one = tf.keras.layers.Activation(activation="sigmoid")(
            tf.keras.layers.Dot(axes=-1)([news_present_one, user_present])
        )
        model = tf.keras.Model([his_input_title, his_input_logw, pred_input_title], preds)
        scorer = tf.keras.Model([his_input_title, his_input_logw, pred_input_title_one], pred_one)
        return model, scorer


def build_variant(ds: dict, variant: str):
    hp = make_hparams(ds["dim"])
    if variant == "baseline":
        return NRMSDocVec(hparams=hp, seed=SEED)
    return NRMSDocVecRecency(hparams=hp, seed=SEED)


_probe_recency = build_variant(datasets[DATASETS[0]], "recency")
print("recency-variant trainable parameters:", f"{_probe_recency.model.count_params():,}")

In [ ]:
def test_recency_layer() -> None:
    rng = np.random.default_rng(0)
    dim = 32
    seq = 6
    vectors = rng.normal(size=(4, seq, dim)).astype(np.float32)

    base = AttLayer2(dim=8, seed=0)
    rec = RecencyAttLayer2(dim=8, seed=0)
    base.build(tf.TensorShape((None, seq, dim)))
    rec.build([tf.TensorShape((None, seq, dim)), tf.TensorShape((None, seq))])
    # Same weights on both layers: this test is about the formula, not the
    # initialiser.
    rec.set_weights(base.get_weights())
    assert len(base.get_weights()) == len(rec.get_weights()) == 3

    zeros = np.zeros((4, seq), dtype=np.float32)
    out_base = base(tf.constant(vectors)).numpy()
    out_zero = rec([tf.constant(vectors), tf.constant(zeros)]).numpy()
    # THE load-bearing assertion: zero log-weights must reproduce the
    # authors' layer exactly, or every "improvement" measured below could be
    # an accidental reimplementation difference instead.
    assert np.array_equal(out_base, out_zero), np.abs(out_base - out_zero).max()

    # Adding a constant to every log-weight cannot change the output: the
    # attention weights are renormalised, so only differences matter.
    shifted = rec([tf.constant(vectors), tf.constant(zeros + 3.5)]).numpy()
    assert np.allclose(out_zero, shifted, atol=1e-5), np.abs(out_zero - shifted).max()

    # A PAD_LOG_WEIGHT slot must contribute exactly nothing: changing that
    # slot's vector leaves the output untouched.
    log_w = np.zeros((4, seq), dtype=np.float32)
    log_w[:, 0] = PAD_LOG_WEIGHT
    out_masked = rec([tf.constant(vectors), tf.constant(log_w)]).numpy()
    tampered = vectors.copy()
    tampered[:, 0, :] = rng.normal(size=(4, dim)).astype(np.float32) * 50.0
    out_tampered = rec([tf.constant(tampered), tf.constant(log_w)]).numpy()
    assert np.allclose(out_masked, out_tampered, atol=1e-6), np.abs(out_masked - out_tampered).max()
    # ... and it must actually differ from attending over the padding.
    assert not np.allclose(out_masked, out_zero, atol=1e-4)

    # A larger log-weight moves attention toward that slot.
    peaked = np.zeros((4, seq), dtype=np.float32)
    peaked[:, -1] = 5.0
    out_peaked = rec([tf.constant(vectors), tf.constant(peaked)]).numpy()
    assert np.abs(out_peaked - vectors[:, -1, :]).mean() < np.abs(out_zero - vectors[:, -1, :]).mean()

    # Identical capacity across variants: the comparison is about the signal,
    # not about parameter count.
    assert _probe.model.count_params() == _probe_recency.model.count_params(), (
        _probe.model.count_params(),
        _probe_recency.model.count_params(),
    )
    assert len(_probe_recency.model.inputs) == 3
    assert tuple(_probe_recency.model.inputs[1].shape) == (None, HISTORY_SIZE)


test_recency_layer()
print("recency layer OK (zero log-weights reproduce ebrec's AttLayer2 exactly, padding excluded, same capacity)")

## Training

One `keras.utils.Sequence` feeds all three variants, differing only in
whether it emits the log-weight tensor and what it puts in it, so no
variant can accidentally train on a different batch order or a different
negative sample.

`metrics=[tf.keras.metrics.AUC(name="auc")]` rather than `metrics=["AUC"]`:
Keras auto-names metrics per session, so the second and third model built
in one kernel get `auc_1`/`auc_2` and `EarlyStopping(monitor="val_auc")`
then silently monitors nothing (it warns and keeps going). Pinning the name
keeps early stopping working for every variant.

In [ ]:
class NRMSSequence(tf.keras.utils.Sequence):
    """Batches (history vectors, [log-weights], candidate vectors) -> one-hot
    labels. Article vectors are gathered per batch from the article matrix,
    never materialised for the whole sample: at HISTORY_SIZE 20 and dim 768
    a fully expanded EB-NeRD training set would be ~27TB."""

    def __init__(self, ds, samples, indices, variant, batch_size, shuffle, seed=SEED):
        self.ds = ds
        self.samples = samples
        self.indices = np.asarray(indices)
        self.variant = variant
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = np.random.default_rng(seed)
        self.order = self.indices.copy()
        if self.shuffle:
            self.order = self.rng.permutation(self.order)

    def __len__(self) -> int:
        return int(math.ceil(len(self.order) / float(self.batch_size)))

    def __getitem__(self, idx):
        rows = self.order[idx * self.batch_size : (idx + 1) * self.batch_size]
        hist_rows = self.samples["hist_row"][rows]
        his = self.ds["art_matrix"][self.ds["hist_idx"][hist_rows]]
        cand = self.ds["art_matrix"][self.samples["cand"][rows]]
        y = self.samples["label"][rows]
        if self.variant == "baseline":
            return [his, cand], y
        log_w = LOG_WEIGHT_FN[self.variant](self.ds, hist_rows, self.samples["impression_us"][rows])
        return [his, log_w, cand], y

    def on_epoch_end(self) -> None:
        if self.shuffle:
            self.order = self.rng.permutation(self.order)


def fit_variant(ds: dict, variant: str, epochs: int = EPOCHS, verbose: int = 2):
    samples = training[ds["name"]]["samples"]
    fit_seq = NRMSSequence(
        ds, samples, training[ds["name"]]["fit_idx"], variant, BATCH_SIZE_TRAIN, shuffle=True
    )
    stop_seq = NRMSSequence(
        ds, samples, training[ds["name"]]["stop_idx"], variant, BATCH_SIZE_TRAIN, shuffle=False
    )
    model_obj = build_variant(ds, variant)
    model_obj.model.compile(
        optimizer=model_obj.model.optimizer,
        loss=model_obj.model.loss,
        metrics=[tf.keras.metrics.AUC(name="auc")],
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc", mode="max", factor=0.2, patience=LR_PLATEAU_PATIENCE, min_lr=1e-6
        ),
    ]
    t0 = time.perf_counter()
    hist = model_obj.model.fit(
        fit_seq, validation_data=stop_seq, epochs=epochs, callbacks=callbacks, verbose=verbose
    )
    seconds = time.perf_counter() - t0
    val_auc = [float(v) for v in hist.history.get("val_auc", [])]
    return model_obj, {
        "epochs_run": len(hist.history.get("loss", [])),
        "loss": [float(v) for v in hist.history.get("loss", [])],
        "val_auc_grouped": val_auc,
        "best_epoch": int(np.argmax(val_auc) + 1) if val_auc else None,
        "fit_seconds": round(seconds, 1),
    }

In [ ]:
def test_training_step() -> None:
    ds = datasets[DATASETS[0]]
    samples = training[ds["name"]]["samples"]
    small = np.arange(min(256, len(samples["cand"])))
    for variant in VARIANTS:
        seq = NRMSSequence(ds, samples, small, variant, 32, shuffle=False)
        (batch, y) = seq[0]
        expected_inputs = 2 if variant == "baseline" else 3
        assert len(batch) == expected_inputs, (variant, len(batch))
        assert batch[0].shape == (32, HISTORY_SIZE, ds["dim"]), batch[0].shape
        assert batch[-1].shape == (32, NPRATIO + 1, ds["dim"]), batch[-1].shape
        assert y.shape == (32, NPRATIO + 1)
        if variant != "baseline":
            assert batch[1].shape == (32, HISTORY_SIZE)
            assert np.isfinite(batch[1]).all()

        # Optimiser steps must reduce the loss on the batch they were taken
        # on: cheap proof that the graph, the inputs and the labels line up
        # for this variant, before hours of fitting. Training-mode loss, from
        # train_on_batch itself, not evaluate() - the news encoder has
        # BatchNormalization, whose inference-mode moving statistics have
        # barely moved after 20 steps, so an evaluate() comparison would be
        # measuring the BN warm-up rather than learning.
        model_obj = build_variant(ds, variant)
        model_obj.model.compile(
            optimizer=model_obj.model.optimizer,
            loss=model_obj.model.loss,
            metrics=[tf.keras.metrics.AUC(name="auc")],
        )
        before = float(np.atleast_1d(model_obj.model.train_on_batch(batch, y))[0])
        for _ in range(20):
            after = float(np.atleast_1d(model_obj.model.train_on_batch(batch, y))[0])
        assert after < before, (variant, before, after)
        preds = model_obj.model.predict(batch, verbose=0)
        assert np.allclose(preds.sum(axis=1), 1.0, atol=1e-5)
        print(f"   {variant}: loss {before:.4f} -> {after:.4f} over 20 steps on one batch")
        del model_obj
    tf.keras.backend.clear_session()


test_training_step()
print("training path OK (batch shapes per variant, loss decreases, softmax intact)")

## Scoring: user vectors once per impression, news vectors once per catalogue

`ebnerd_nrms_docvec.py` scores with `model.scorer.predict(dataloader)`,
which flattens every (impression, candidate) pair into its own row and so
re-encodes the user's whole 20-click history once **per candidate**. On this
evaluation population that is 200,000 impressions x ~12 (EB-NeRD) to ~37
(MIND) candidates x 20 history slots of news-encoder work per split, and it
is redundant: `NRMSDocVec` scores by a dot product between a user vector
that depends only on the history and a news vector that depends only on the
article.

So the user vector is computed once per impression, the news vectors once
for the whole catalogue, and the score is their dot product - the same
number the scorer's final `Dot` layer produces, before a sigmoid that
cannot reorder anything. The test cell checks that equivalence against
`model.scorer.predict` directly, and reports the measured speedup.

The improvement keeps this property, which is part of why it was chosen:
a candidate-aware user encoder would make the user vector depend on the
candidate and cost the full 12x-37x again at serving time.

In [ ]:
def news_vectors(model_obj, ds: dict) -> np.ndarray:
    """Encode the whole article matrix once, padding row included."""
    return model_obj.newsencoder.predict(ds["art_matrix"], batch_size=NEWS_BATCH, verbose=0)


def user_vectors(model_obj, ds: dict, split: str, variant: str, limit: int | None = None) -> np.ndarray:
    s = ds["splits"][split]
    n = s["n_impressions"] if limit is None else min(limit, s["n_impressions"])
    out = None
    for start in range(0, n, USER_BATCH):
        end = min(start + USER_BATCH, n)
        hist_rows = s["hist_row"][start:end]
        his = ds["art_matrix"][ds["hist_idx"][hist_rows]]
        if variant == "baseline":
            batch_out = model_obj.userencoder.predict(his, batch_size=USER_BATCH, verbose=0)
        else:
            log_w = LOG_WEIGHT_FN[variant](ds, hist_rows, s["impression_us"][start:end])
            batch_out = model_obj.userencoder.predict(
                [his, log_w], batch_size=USER_BATCH, verbose=0
            )
        if out is None:
            out = np.empty((n, batch_out.shape[1]), dtype=np.float32)
        out[start:end] = batch_out
    return out


def scores_for_impression(news_vecs: np.ndarray, user_vecs: np.ndarray, s: dict, i: int) -> np.ndarray:
    lo, hi = s["offsets"][i], s["offsets"][i + 1]
    return news_vecs[s["cand"][lo:hi]] @ user_vecs[i]

In [ ]:
def test_scoring_equivalence() -> None:
    ds = datasets[DATASETS[0]]
    split = "val"
    s = ds["splits"][split]
    # Kept small deliberately: the reference path expands the history once
    # per candidate, so 128 MIND impressions already materialise ~4,700 x 20
    # x 768 floats.
    n_probe = min(128, s["n_impressions"])
    for variant in ("baseline", "recency"):
        model_obj = build_variant(ds, variant)
        news_vecs = news_vectors(model_obj, ds)
        assert news_vecs.shape[0] == ds["art_matrix"].shape[0]

        t0 = time.perf_counter()
        user_vecs = user_vectors(model_obj, ds, split, variant, limit=n_probe)
        fast_scores = np.concatenate(
            [scores_for_impression(news_vecs, user_vecs, s, i) for i in range(n_probe)]
        )
        fast_seconds = time.perf_counter() - t0

        # The authors' scoring path: one row per (impression, candidate), the
        # history re-encoded for each.
        rows = []
        for i in range(n_probe):
            lo, hi = s["offsets"][i], s["offsets"][i + 1]
            rows.append((i, s["cand"][lo:hi]))
        his_rep = np.concatenate(
            [np.repeat(ds["hist_idx"][s["hist_row"][i]][None, :], len(c), axis=0) for i, c in rows]
        )
        cand_rep = np.concatenate([c for _, c in rows])
        his_ref = ds["art_matrix"][his_rep]
        cand_ref = ds["art_matrix"][cand_rep][:, None, :]
        t0 = time.perf_counter()
        if variant == "baseline":
            ref = model_obj.scorer.predict([his_ref, cand_ref], batch_size=512, verbose=0).ravel()
        else:
            logw_rep = np.concatenate(
                [
                    LOG_WEIGHT_FN[variant](
                        ds,
                        np.repeat(s["hist_row"][i], len(c)),
                        np.repeat(s["impression_us"][i], len(c)),
                    )
                    for i, c in rows
                ]
            )
            ref = model_obj.scorer.predict(
                [his_ref, logw_rep, cand_ref], batch_size=512, verbose=0
            ).ravel()
        ref_seconds = time.perf_counter() - t0

        sigmoid = 1.0 / (1.0 + np.exp(-fast_scores.astype(np.float64)))
        assert np.allclose(sigmoid, ref, atol=1e-4), np.abs(sigmoid - ref).max()
        # Ranking is what the metrics actually see. Impressions where the
        # scorer's sigmoid has saturated are skipped: there the *scorer* has
        # lost the ordering (several candidates come back as exactly 1.0),
        # not the shortcut, so a comparison would be testing float64's
        # resolution near 1. Elsewhere a reordering can only happen where two
        # candidates sit closer together than float32 noise (~1e-6), which
        # cannot move a reported metric materially, so a single flip is
        # tolerated and counted rather than ignored.
        flips = 0
        saturated = 0
        for i in range(n_probe):
            lo, hi = s["offsets"][i], s["offsets"][i + 1]
            window = ref[lo:hi]
            if window.max() > 1.0 - 1e-9 or window.min() < 1e-9:
                saturated += 1
                continue
            a = np.argsort(-fast_scores[lo:hi], kind="stable")
            b = np.argsort(-window, kind="stable")
            if not np.array_equal(a, b):
                flips += 1
        assert saturated < n_probe, f"{variant}: every probe impression saturated the scorer"
        assert flips <= max(1, n_probe // 100), (variant, flips, n_probe, saturated)
        print(
            f"   {variant}: {n_probe} impressions, max |sigmoid(dot) - scorer| "
            f"{np.abs(sigmoid - ref).max():.2e}, {flips} ranking flips "
            f"({saturated} skipped as saturated), "
            f"{ref_seconds / max(fast_seconds, 1e-9):.1f}x faster on the scoring step "
            f"({fast_seconds:.2f}s vs {ref_seconds:.2f}s; the one-off catalogue encoding is "
            "excluded from both and amortises over every impression of both splits)"
        )
        del model_obj, news_vecs, user_vecs
        tf.keras.backend.clear_session()


test_scoring_equivalence()
print("scoring path OK (identical rankings to ebrec's scorer, at a fraction of the work)")

## Metrics

`auc_impression`, `mrr`, `ndcg_at_k`, `bootstrap_ci` and
`paired_bootstrap_ci` are the local package's implementations
(`src/cs4406m26_assignment1c1/evaluation.py`) restated here, because Kaggle
has no access to the repo. They must stay identical: A1 and A2 Q2's numbers
were produced by them, and Q3's tables are read against those. The test
cell pins them to hand-computed values, including the two tie-handling
details that are easy to get silently wrong (average ranks for AUC ties,
stable argsort elsewhere) and the case where nDCG@10 comes out *below*
nDCG@5.

In [ ]:
def _rank_avg(scores: np.ndarray) -> np.ndarray:
    order = np.argsort(scores, kind="stable")
    sorted_scores = scores[order]
    ranks = np.empty(len(scores), dtype=np.float64)
    n = len(scores)
    i = 0
    while i < n:
        j = i
        while j + 1 < n and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        ranks[order[i : j + 1]] = (i + 1 + j + 1) / 2.0
        i = j + 1
    return ranks


def auc_impression(scores, labels) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=bool)
    n_pos = int(labels.sum())
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        raise ValueError("AUC is undefined without both a click and a non-click in the inview set")
    ranks = _rank_avg(scores)
    rank_sum_pos = ranks[labels].sum()
    return (rank_sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def mrr(scores, labels) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=bool)
    order = np.argsort(-scores, kind="stable")
    hit_positions = np.flatnonzero(labels[order])
    if len(hit_positions) == 0:
        raise ValueError("MRR is undefined without at least one click in the inview set")
    return 1.0 / (hit_positions[0] + 1)


def ndcg_at_k(scores, labels, k: int) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    order = np.argsort(-scores, kind="stable")
    ranked_labels = labels[order][:k]
    dcg = float((ranked_labels / np.log2(np.arange(2, len(ranked_labels) + 2))).sum())
    n_pos = int(labels.sum())
    ideal_len = min(n_pos, k)
    if ideal_len == 0:
        raise ValueError("nDCG is undefined without at least one click in the inview set")
    idcg = float((1.0 / np.log2(np.arange(2, ideal_len + 2))).sum())
    return dcg / idcg


def bootstrap_ci(values, n_iterations=BOOTSTRAP_ITERATIONS, seed=BOOTSTRAP_SEED, max_chunk_cells=200_000_000):
    values = np.asarray(values, dtype=np.float64)
    n = len(values)
    if n == 0:
        raise ValueError("bootstrap_ci is undefined for an empty slice")
    point = float(values.mean())
    if n == 1:
        return point, point, point
    rng = np.random.default_rng(seed)
    iters_per_chunk = max(1, max_chunk_cells // n)
    resample_means = np.empty(n_iterations, dtype=np.float64)
    start = 0
    while start < n_iterations:
        end = min(start + iters_per_chunk, n_iterations)
        resample_idx = rng.integers(0, n, size=(end - start, n), dtype=np.int32)
        resample_means[start:end] = values[resample_idx].mean(axis=1)
        start = end
    lo, hi = np.percentile(resample_means, [2.5, 97.5])
    return point, float(lo), float(hi)


def paired_bootstrap_ci(
    values_a, values_b, n_iterations=BOOTSTRAP_ITERATIONS, seed=BOOTSTRAP_SEED, max_chunk_cells=200_000_000
):
    """Bootstrap `mean(b) - mean(a)` over the *same* impressions in the same
    order. Two independent CIs that happen to overlap do not imply the
    difference is insignificant: per-impression difficulty is shared between
    the variants, and resampling one index array and applying it to both is
    what makes that shared variance cancel."""
    values_a = np.asarray(values_a, dtype=np.float64)
    values_b = np.asarray(values_b, dtype=np.float64)
    if values_a.shape != values_b.shape:
        raise ValueError(
            f"paired_bootstrap_ci needs the same impressions in both arrays, "
            f"got {values_a.shape} and {values_b.shape}"
        )
    n = len(values_a)
    if n == 0:
        raise ValueError("paired_bootstrap_ci is undefined for an empty slice")
    diff = values_b - values_a
    point = float(diff.mean())
    if n == 1:
        return point, point, point
    rng = np.random.default_rng(seed)
    iters_per_chunk = max(1, max_chunk_cells // n)
    resample_means = np.empty(n_iterations, dtype=np.float64)
    start = 0
    while start < n_iterations:
        end = min(start + iters_per_chunk, n_iterations)
        resample_idx = rng.integers(0, n, size=(end - start, n), dtype=np.int32)
        resample_means[start:end] = diff[resample_idx].mean(axis=1)
        start = end
    lo, hi = np.percentile(resample_means, [2.5, 97.5])
    return point, float(lo), float(hi)


def per_impression_metrics(news_vecs, user_vecs, ds, split) -> dict[str, np.ndarray]:
    s = ds["splits"][split]
    n = s["n_impressions"]
    out = {name: np.empty(n, dtype=np.float64) for name in METRIC_NAMES}
    for i in range(n):
        lo, hi = s["offsets"][i], s["offsets"][i + 1]
        scores = news_vecs[s["cand"][lo:hi]] @ user_vecs[i]
        labels = s["label"][lo:hi]
        out["auc"][i] = auc_impression(scores, labels)
        out["mrr"][i] = mrr(scores, labels)
        for k in NDCG_K_VALUES:
            out[f"ndcg{k}"][i] = ndcg_at_k(scores, labels, k)
    return out

In [ ]:
def test_metrics() -> None:
    # AUC: perfect, inverted, all-tied, and a two-positive case.
    assert auc_impression([0.9, 0.1], [1, 0]) == 1.0
    assert auc_impression([0.1, 0.9], [1, 0]) == 0.0
    assert auc_impression([0.5, 0.5, 0.5], [1, 0, 1]) == 0.5
    assert auc_impression([0.9, 0.8, 0.1], [1, 1, 0]) == 1.0
    try:
        auc_impression([0.9, 0.1], [1, 1])
        raise AssertionError("AUC must be undefined without a negative")
    except ValueError:
        pass

    # MRR: 1, 1/2, 1/3, and multiple clicks take the first hit.
    assert mrr([0.9, 0.5, 0.1], [1, 0, 0]) == 1.0
    assert mrr([0.9, 0.5, 0.1], [0, 1, 0]) == 0.5
    assert abs(mrr([0.9, 0.5, 0.1], [0, 0, 1]) - 1 / 3) < 1e-12
    assert mrr([0.9, 0.5, 0.1], [0, 1, 1]) == 0.5

    # nDCG: a perfect ranking is 1.0, and the hand-computed value for a
    # single positive at rank 2 is 1/log2(3).
    assert abs(ndcg_at_k([0.9, 0.5, 0.1], [1, 0, 0], 5) - 1.0) < 1e-12
    assert abs(ndcg_at_k([0.5, 0.9, 0.1], [1, 0, 0], 5) - 1.0 / math.log2(3)) < 1e-12

    # nDCG@10 below nDCG@5, which is not a bug (A2 Q2 #11): IDCG@k sums
    # min(n_pos, k) terms, so the denominator can grow faster than the
    # numerator. Six positives at ranks 1-5 and 7 of eight candidates.
    scores = [8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]
    labels = [1, 1, 1, 1, 1, 0, 1, 0]
    assert abs(ndcg_at_k(scores, labels, 5) - 1.0) < 1e-12
    assert abs(ndcg_at_k(scores, labels, 10) - 0.993078) < 1e-5, ndcg_at_k(scores, labels, 10)
    assert ndcg_at_k(scores, labels, 10) < ndcg_at_k(scores, labels, 5)

    # Bootstrap: a known +0.02 effect with shared per-impression difficulty.
    rng = np.random.default_rng(7)
    difficulty = rng.normal(size=20000) * 0.2
    a = 0.60 + difficulty + rng.normal(size=20000) * 0.02
    b = a + 0.02 + rng.normal(size=20000) * 0.02
    diff, lo, hi = paired_bootstrap_ci(a, b, 1000, 0)
    assert lo > 0 and hi > lo and lo < 0.02 < hi, (diff, lo, hi)
    # The paired interval must be much tighter than an unpaired one on the
    # same data, which is the reason for pairing at all.
    _, lo_a, hi_a = bootstrap_ci(a, 1000, 0)
    _, lo_b, hi_b = bootstrap_ci(b, 1000, 0)
    unpaired_width = (hi_b - lo_b) + (hi_a - lo_a)
    assert (hi - lo) * 4 < unpaired_width, ((hi - lo), unpaired_width)
    # No effect -> the interval covers zero.
    _, lo0, hi0 = paired_bootstrap_ci(a, a + rng.normal(size=20000) * 0.02, 1000, 0)
    assert lo0 < 0 < hi0, (lo0, hi0)
    # Chunked equals unchunked exactly at a fixed seed.
    full = paired_bootstrap_ci(a, b, 100, 0, max_chunk_cells=10**12)
    chunked = paired_bootstrap_ci(a, b, 100, 0, max_chunk_cells=20000 * 7)
    assert full == chunked, (full, chunked)
    try:
        paired_bootstrap_ci(a, b[:10])
        raise AssertionError("a shape mismatch must raise")
    except ValueError:
        pass

    # Cross-check the mean AUC against ebrec's own metric implementation on a
    # synthetic population, so the reproduction is not judged by our
    # definition alone. Skipped rather than failed if their evaluation module
    # cannot import in this environment.
    rng = np.random.default_rng(3)
    y_true = [[1, 0, 0, 0], [0, 1, 0], [1, 0, 1, 0, 0]]
    y_pred = [list(rng.normal(size=len(row))) for row in y_true]
    their_auc = None
    try:
        from ebrec.evaluation import AucScore, MetricEvaluator

        theirs = MetricEvaluator(labels=y_true, predictions=y_pred, metric_functions=[AucScore()]).evaluate()
        values = theirs.evaluations if hasattr(theirs, "evaluations") else theirs
        their_auc = float(list(dict(values).values())[0])
    except Exception as exc:  # noqa: BLE001
        print(f"   skipped the ebrec metric cross-check ({type(exc).__name__}: {exc})")
    # Asserted outside the try, so a real disagreement fails instead of being
    # caught and reported as a skip.
    if their_auc is not None:
        ours = float(np.mean([auc_impression(p, t) for p, t in zip(y_pred, y_true)]))
        assert abs(their_auc - ours) < 1e-9, (their_auc, ours)
        print(f"   ebrec's AucScore agrees with auc_impression: {their_auc:.6f}")


test_metrics()
print("metrics OK (hand-computed AUC/MRR/nDCG, paired CI tighter than unpaired, chunking exact)")

## Fit and score every variant

One variant at a time, and each is released before the next is built:
three models plus a 125,541 x 768 article matrix and the encoded
catalogue do not need to be resident simultaneously, and A1's memory work
(SPEC.md Q4 #9, A2 Q1 #8) is a long record of what happens when
separately-affordable transients are held at once.

Per-impression metric arrays are kept - not just their means - because the
paired bootstrap needs them, and they are indexed identically across
variants by construction: the same `offsets` array, in the same order, for
every variant.

In [ ]:
per_impression: dict[tuple[str, str, str], dict[str, np.ndarray]] = {}
train_reports: dict[tuple[str, str], dict] = {}

for name in DATASETS:
    ds = datasets[name]
    for variant in VARIANTS:
        print(f"\n=== {name} / {variant} " + "=" * 40)
        model_obj, report = fit_variant(ds, variant)
        weights_path = WORK / f"nrms_{variant}_{name}.weights.h5"
        model_obj.model.save_weights(str(weights_path))
        report["weights"] = weights_path.name
        report["parameters"] = int(model_obj.model.count_params())
        train_reports[(name, variant)] = report

        t0 = time.perf_counter()
        news_vecs = news_vectors(model_obj, ds)
        for split in SPLITS:
            user_vecs = user_vectors(model_obj, ds, split, variant)
            per_impression[(name, variant, split)] = per_impression_metrics(news_vecs, user_vecs, ds, split)
            means = {m: float(per_impression[(name, variant, split)][m].mean()) for m in METRIC_NAMES}
            print(f"   {split}: " + "  ".join(f"{m}={v:.4f}" for m, v in means.items()))
            del user_vecs
        report["score_seconds"] = round(time.perf_counter() - t0, 1)
        del model_obj, news_vecs
        tf.keras.backend.clear_session()

{f"{n}/{v}": train_reports[(n, v)]["fit_seconds"] for n in DATASETS for v in VARIANTS}

In [ ]:
def test_scored_population() -> None:
    for name in DATASETS:
        for split in SPLITS:
            n = datasets[name]["splits"][split]["n_impressions"]
            for variant in VARIANTS:
                per = per_impression[(name, variant, split)]
                for metric in METRIC_NAMES:
                    values = per[metric]
                    # The paired CI is only meaningful if position i means the
                    # same impression for every variant; equal lengths over a
                    # shared offsets array is what guarantees that here.
                    assert len(values) == n, (name, variant, split, metric, len(values), n)
                    assert np.isfinite(values).all(), f"{name}/{variant}/{split}/{metric}: non-finite"
                    assert values.min() >= 0.0 and values.max() <= 1.0, (metric, values.min(), values.max())
                # nDCG@10 may sit below nDCG@5 only where an impression has
                # more than 5 clicks; where it does not, the order must hold.
                n_pos = np.add.reduceat(
                    datasets[name]["splits"][split]["label"].astype(np.int64),
                    datasets[name]["splits"][split]["offsets"][:-1],
                )
                few = n_pos <= 5
                assert np.all(per["ndcg10"][few] >= per["ndcg5"][few] - 1e-12), (name, variant, split)
            # Different variants must actually produce different rankings; if
            # they agreed exactly, something is feeding the same model twice.
            base = per_impression[(name, "baseline", split)]["auc"]
            rec = per_impression[(name, "recency", split)]["auc"]
            assert not np.array_equal(base, rec), f"{name}/{split}: baseline and recency scored identically"


test_scored_population()
print("scored population OK (aligned across variants, bounded, distinct)")

## Results, ablation and paired significance

`nrms_metrics_{dataset}.json` mirrors the shape of A2 Q2's
`reranker_eval_metrics.json` on purpose - `ranking_metrics`,
`paired_comparison`, `hyperparameters`, `population` - so the two can be
read by the same code and quoted in the same table.

Three paired comparisons per split:

| comparison | question |
|---|---|
| `baseline_vs_recency` | Does the improved model beat the reproduced baseline? |
| `masked_uniform_vs_recency` | How much of that is the recency signal itself? |
| `baseline_vs_masked_uniform` | How much is masking the padded history slots? |

A claimed gain counts only if `excludes_zero` is true for it. The point
estimates decompose exactly - the first difference is the sum of the other
two, since means are linear - which the test cell asserts as an internal
consistency check on the ablation.

In [ ]:
PAIRS = [
    ("baseline", "recency"),
    ("masked_uniform", "recency"),
    ("baseline", "masked_uniform"),
]


def build_metrics_payload(name: str) -> dict:
    ds = datasets[name]
    ranking = {}
    for variant in VARIANTS:
        ranking[variant] = {}
        for split in SPLITS:
            per = per_impression[(name, variant, split)]
            ranking[variant][split] = {
                metric: dict(zip(("point", "ci_lo", "ci_hi"), bootstrap_ci(per[metric])))
                for metric in METRIC_NAMES
            }

    paired = {}
    for a, b in PAIRS:
        entry = paired.setdefault(f"{a}_vs_{b}", {})
        for split in SPLITS:
            per_metric = {}
            for metric in METRIC_NAMES:
                diff, lo, hi = paired_bootstrap_ci(
                    per_impression[(name, a, split)][metric],
                    per_impression[(name, b, split)][metric],
                )
                per_metric[metric] = {
                    "mean_diff": diff,
                    "ci_lo": lo,
                    "ci_hi": hi,
                    "excludes_zero": bool(lo > 0 or hi < 0),
                }
            entry[split] = per_metric

    return {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "dataset": name,
        "baseline": {
            "source": EBNERD_REPO_URL,
            "commit": EBNERD_COMMIT,
            "model": "NRMSDocVec",
            "code_modified": False,
            "hparams": hparams_to_dict(make_hparams(ds["dim"])),
        },
        "improvement": {
            "name": "recency-weighted history attention",
            "recency_weight_basis": ds["basis"],
            "half_life": ds["half_life"],
            "half_life_unit": "hours" if ds["basis"] == "elapsed_time" else "clicks",
            "changed_component": "user_encoder additive attention (AttLayer2 -> RecencyAttLayer2)",
            "added_parameters": 0,
        },
        "hyperparameters": {
            "history_size": HISTORY_SIZE,
            "npratio": NPRATIO,
            "batch_size": BATCH_SIZE_TRAIN,
            "epochs": EPOCHS,
            "seed": SEED,
            "ndcg_k_values": NDCG_K_VALUES,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
            "bootstrap_seed": BOOTSTRAP_SEED,
        },
        "population": {
            "train_samples": int(len(training[name]["samples"]["cand"])),
            "fit_samples": int(len(training[name]["fit_idx"])),
            "early_stopping_samples": int(len(training[name]["stop_idx"])),
            **{
                split: {
                    "impressions": ds["splits"][split]["n_impressions"],
                    "candidates": int(len(ds["splits"][split]["cand"])),
                    "positive_rate": float(ds["splits"][split]["label"].mean()),
                }
                for split in SPLITS
            },
        },
        "training": {variant: train_reports[(name, variant)] for variant in VARIANTS},
        "ranking_metrics": ranking,
        "paired_comparison": paired,
    }


payloads = {}
for name in DATASETS:
    payload = build_metrics_payload(name)
    payloads[name] = payload
    path = WORK / f"nrms_metrics_{name}.json"
    path.write_text(json.dumps(payload, indent=2, default=str))

    print(f"\n### {name} ".ljust(72, "#"))
    header = "variant".ljust(16) + "split".ljust(7) + "".join(m.ljust(24) for m in METRIC_NAMES)
    print(header)
    for variant in VARIANTS:
        for split in SPLITS:
            cells = []
            for metric in METRIC_NAMES:
                e = payload["ranking_metrics"][variant][split][metric]
                cells.append(f"{e['point']:.4f} [{e['ci_lo']:.4f},{e['ci_hi']:.4f}]".ljust(24))
            print(variant.ljust(16) + split.ljust(7) + "".join(cells))
    print("\npaired 95% CI on the difference (b - a), 1000 iterations, identical impressions:")
    for key, per_split in payload["paired_comparison"].items():
        for split in SPLITS:
            e = per_split[split]["auc"]
            flag = "excludes zero" if e["excludes_zero"] else "INCLUDES ZERO"
            print(
                f"   {key:<34} {split:<5} d_auc={e['mean_diff']:+.4f} "
                f"[{e['ci_lo']:+.4f}, {e['ci_hi']:+.4f}]  {flag}"
            )

sorted(str(p) for p in WORK.glob("nrms_metrics_*.json"))

In [ ]:
def test_payloads() -> None:
    for name in DATASETS:
        payload = json.loads((WORK / f"nrms_metrics_{name}.json").read_text())
        assert payload["baseline"]["commit"] == EBNERD_COMMIT
        assert payload["improvement"]["added_parameters"] == 0
        params = {payload["training"][v]["parameters"] for v in VARIANTS}
        assert len(params) == 1, f"{name}: variants differ in parameter count {params}"

        for variant in VARIANTS:
            for split in SPLITS:
                for metric in METRIC_NAMES:
                    e = payload["ranking_metrics"][variant][split][metric]
                    # Tolerance because the point estimate is the sample mean
                    # while the bounds are resample percentiles: they bracket
                    # it in every realistic case but are not guaranteed to.
                    assert 0.0 <= e["ci_lo"] <= e["ci_hi"] <= 1.0, (name, variant, split, metric, e)
                    assert e["ci_lo"] - 1e-6 <= e["point"] <= e["ci_hi"] + 1e-6, (
                        name,
                        variant,
                        split,
                        metric,
                        e,
                    )

        for key, per_split in payload["paired_comparison"].items():
            for split in SPLITS:
                for metric in METRIC_NAMES:
                    e = per_split[split][metric]
                    assert e["ci_lo"] - 1e-6 <= e["mean_diff"] <= e["ci_hi"] + 1e-6, (
                        name,
                        key,
                        split,
                        metric,
                        e,
                    )
                    assert e["excludes_zero"] == (e["ci_lo"] > 0 or e["ci_hi"] < 0)

        # The ablation has to decompose: baseline -> recency is exactly
        # (baseline -> masked_uniform) + (masked_uniform -> recency), because
        # every term is a mean over the same impressions.
        for split in SPLITS:
            for metric in METRIC_NAMES:
                total = payload["paired_comparison"]["baseline_vs_recency"][split][metric]["mean_diff"]
                mask = payload["paired_comparison"]["baseline_vs_masked_uniform"][split][metric]["mean_diff"]
                recency = payload["paired_comparison"]["masked_uniform_vs_recency"][split][metric]["mean_diff"]
                assert abs(total - (mask + recency)) < 1e-9, (name, split, metric, total, mask, recency)


test_payloads()
print("metrics payload OK (CIs ordered, flags consistent, ablation decomposes exactly)")

## Plots

In [ ]:
import matplotlib.pyplot as plt

BG = "#121212"
FG = "#E0E0E0"
MUTED = "#B0BEC5"
GRID = "#2A2A2A"
EDGE = "#37474F"
VARIANT_COLORS = {"baseline": "#B0BEC5", "masked_uniform": "#FFB74D", "recency": "#4FC3F7"}


def style(ax) -> None:
    ax.set_facecolor(BG)
    ax.tick_params(colors=MUTED)
    for spine in ax.spines.values():
        spine.set_color(EDGE)
    ax.grid(True, color=GRID, linestyle="--", linewidth=0.7, axis="y")


fig, axes = plt.subplots(
    1, len(DATASETS), figsize=(6.5 * len(DATASETS), 4.4), facecolor=BG, squeeze=False
)
for ax, name in zip(axes[0], DATASETS):
    style(ax)
    labels = [f"{split}" for split in SPLITS]
    x = np.arange(len(SPLITS))
    width = 0.26
    for i, variant in enumerate(VARIANTS):
        points, errs = [], [[], []]
        for split in SPLITS:
            e = payloads[name]["ranking_metrics"][variant][split]["auc"]
            points.append(e["point"])
            errs[0].append(e["point"] - e["ci_lo"])
            errs[1].append(e["ci_hi"] - e["point"])
        ax.bar(
            x + (i - 1) * width,
            points,
            width,
            label=variant,
            color=VARIANT_COLORS[variant],
            yerr=errs,
            ecolor=FG,
            capsize=3,
        )
        for xi, p in zip(x + (i - 1) * width, points):
            ax.annotate(
                f"{p:.4f}", (xi, p), textcoords="offset points", xytext=(0, 6), ha="center",
                color=FG, fontsize=7.5,
            )
    ax.axhline(0.5, color="#EF5350", linestyle=":", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, color=FG)
    ax.set_ylabel("AUC per impression", color=FG)
    ax.set_title(f"{name} ({datasets[name]['basis']})", color="#FFFFFF")
    ax.set_ylim(0.45, max(0.75, ax.get_ylim()[1]))
    ax.legend(facecolor=BG, edgecolor=EDGE, labelcolor=FG, fontsize=8)
fig.suptitle("NRMS ablation: baseline, padding-masked, recency-weighted", color="#FFFFFF")
fig.tight_layout()
fig.savefig(WORK / "nrms_ablation.png", dpi=150, facecolor=BG)
plt.show()

fig, axes = plt.subplots(
    1, len(DATASETS), figsize=(6.5 * len(DATASETS), 4.4), facecolor=BG, squeeze=False
)
for ax, name in zip(axes[0], DATASETS):
    style(ax)
    rows = [(key, split) for key in payloads[name]["paired_comparison"] for split in SPLITS]
    ys = np.arange(len(rows))
    for y, (key, split) in zip(ys, rows):
        e = payloads[name]["paired_comparison"][key][split]["auc"]
        color = "#4FC3F7" if e["excludes_zero"] else "#EF5350"
        ax.errorbar(
            e["mean_diff"], y,
            xerr=[[e["mean_diff"] - e["ci_lo"]], [e["ci_hi"] - e["mean_diff"]]],
            fmt="o", color=color, ecolor=color, capsize=3, markersize=5,
        )
    ax.axvline(0.0, color="#EF5350", linestyle=":", linewidth=1.2)
    ax.set_yticks(ys)
    ax.set_yticklabels([f"{k.replace('_vs_', ' -> ')} / {s}" for k, s in rows], color=FG, fontsize=8)
    ax.set_xlabel("delta AUC (paired bootstrap 95% CI)", color=FG)
    ax.set_title(name, color="#FFFFFF")
    ax.grid(True, color=GRID, linestyle="--", linewidth=0.7, axis="x")
fig.suptitle("Paired bootstrap CIs on the AUC difference (blue excludes zero)", color="#FFFFFF")
fig.tight_layout()
fig.savefig(WORK / "nrms_paired_ci.png", dpi=150, facecolor=BG)
plt.show()

In [ ]:
def test_plots() -> None:
    for fname in ("nrms_ablation.png", "nrms_paired_ci.png"):
        path = WORK / fname
        assert path.exists() and path.stat().st_size > 10_000, f"{fname} was not written"
    for name in DATASETS:
        assert (WORK / f"nrms_metrics_{name}.json").exists()
        for variant in VARIANTS:
            assert (WORK / f"nrms_{variant}_{name}.weights.h5").exists()


test_plots()
print("outputs OK:", sorted(p.name for p in WORK.glob("nrms_*")))

# Manual Verification Complete

Download from `/kaggle/working/`:

- `nrms_metrics_{dataset}.json` -> `data/processed/{dataset}/`
- `nrms_{variant}_{dataset}.weights.h5` -> `data/processed/{dataset}/`
  (gitignored, per the A2 git policy)
- `nrms_ablation.png`, `nrms_paired_ci.png` -> repo root, for the design
  note

What the JSON supports, and what it does not: `ranking_metrics` is directly
comparable to `reranker_eval_metrics.json`, because both were measured on
the same impressions with the same estimators. `paired_comparison` is the
evidence for Q3's items 2-4 - a gain counts only where `excludes_zero` is
true, and the design note quotes the interval, not just the point estimate.